# VAE vs supervised VAE

Same architecture, same data, same schedule -- the only difference is three linear
classification heads reading three blocks of the latent during training. This notebook
asks what that bought and what it cost.

**The cost side.** Reconstruction MSE, the two ELBO terms (per-image BCE sum, KL in nats),
whether reconstructions still carry the right colours, and whether the digit judge still
reads the right digit off them. `digit/real` is the judge on the *inputs*: the ceiling
`digit/recon` is chasing, not 1.0.

**The structure side.** Three questions a single probe cannot separate:

| | question | probe |
|---|---|---|
| **sufficiency** | is the factor in the latent at all? | every dimension |
| **locality** | is it in the block it was assigned? | that block alone |
| **exclusivity** | is it *only* there? | every other dimension |

Linear probes throughout, because "linearly decodable from its own block" is what the
heads are trained for and what a circuit downstream can exploit. The MLP probe sits beside
it to separate *not represented* from *represented nonlinearly* -- the unsupervised
latent's digit sits at roughly 0.5 linear against 0.97 MLP, which is the gap supervision
is meant to close.

A supervised block has done its job when **locality is high and leakage is low**. High
locality alone is not enough: a factor can be readable from its block *and* from
everywhere else.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from torchvision import transforms

from dataset_loaders.colour_mnist import ColourMNIST
from evaluation import (
    blocks_for,
    encode_dataset,
    latent_traversal,
    load_digit_classifier,
    probe_latents,
    reconstruct,
    reconstruction_summary,
)
from utils import resolve_device
from utils.checkpoints import load_ae_from_path
from utils.visualisation import plot_image_rows
from utils.wandb_utils import load_from_wandb

# Artifact per model. The blocks a supervised model was trained with come from its own
# config; FALLBACK_BLOCKS is what an unsupervised latent gets probed on, so both models
# are asked about the same dimension ranges.
MODELS = {
    "VAE": "variational_colour_mnist_uniform",
    "supervised VAE": "supervised_colour_mnist_uniform",
}
TAG = "latest"
LOCAL_CHECKPOINTS: dict[str, str] = {}  # name -> path, to bypass wandb

VARIANT = "uniform"
FALLBACK_BLOCKS = [slice(0, 5), slice(5, 8), slice(8, 10)]
FACTOR_NAMES = ["digit", "fg", "bg"]

PROBE_TRAIN, PROBE_TEST = 20_000, 10_000
BATCH_SIZE = 256
FIGURES = Path("artifacts/figures")
FIGURES.mkdir(parents=True, exist_ok=True)

device = resolve_device()
device

In [ ]:
def autoencoder(name: str):
    path = LOCAL_CHECKPOINTS.get(name) or load_from_wandb(MODELS[name], TAG)
    return load_ae_from_path(path, device=device).to(device).eval()


def loader(split: str, limit: int | None = None) -> DataLoader:
    dataset = ColourMNIST(
        root="../data",
        split=split,
        variant=VARIANT,
        transform=transforms.Compose([transforms.ToTensor()]),
    )
    if limit is not None:
        dataset = torch.utils.data.Subset(dataset, range(min(limit, len(dataset))))
    return DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)


models = {name: autoencoder(name) for name in MODELS}
judge = load_digit_classifier(device=device)
{name: type(model).__name__ for name, model in models.items()}

## Cost: what supervision gives up

In [ ]:
costs = pd.DataFrame(
    {
        name: reconstruction_summary(model, loader("test"), device, classifier=judge)
        for name, model in models.items()
    }
).loc[["mse", "bce", "kl", "elbo", "fg_accuracy", "bg_accuracy", "digit_recon", "digit_real"]]

costs["delta"] = costs.iloc[:, 1] - costs.iloc[:, 0]
costs.round(4)

## Structure: sufficiency, locality, exclusivity

The probes fit on the train split and score on test, so a high number is generalisation,
not memorisation.

In [ ]:
probes = {}
kls = {}
latents = {}
for name, model in models.items():
    train_z, train_kl, train_y = encode_dataset(model, loader("train", PROBE_TRAIN), device)
    test_z, _, test_y = encode_dataset(model, loader("test", PROBE_TEST), device)
    blocks, names = blocks_for(model, FALLBACK_BLOCKS, FACTOR_NAMES)
    probes[name] = probe_latents(
        train_z, train_y, test_z, test_y, blocks, names, kl_per_dim=train_kl
    )
    kls[name] = train_kl
    latents[name] = train_z  # kept for the traversal sweep range below
    print(f"{name}: {probes[name].active_units} of {train_z.shape[1]} dimensions active")

table = pd.concat({name: report.to_frame() for name, report in probes.items()})
table.round(3)

In [ ]:
# The comparison in one view: locality should rise, leakage should fall.
headline = pd.DataFrame(
    {
        name: {
                  f"{probe.name} · locality": probe.locality_linear
                  for probe in report.factors
              }
              | {f"{probe.name} · leakage": probe.exclusivity_linear for probe in report.factors}
        for name, report in probes.items()
    }
)
headline.round(3)

In [ ]:
figure, ax = plt.subplots(figsize=(9, 3), dpi=140)
width = 0.8 / len(kls)
for offset, (name, kl) in enumerate(kls.items()):
    ax.bar(np.arange(len(kl)) + offset * width, kl, width=width, label=name)
ax.set_xlabel("latent dimension")
ax.set_ylabel("KL (nats)")
ax.set_title("per-dimension KL — a dead dimension carries nothing to marginalise")
ax.legend()
figure.tight_layout()
figure.savefig(FIGURES / "vae_kl_per_dim.png", bbox_inches="tight")

## Reconstructions

In [ ]:
COUNT = 10
originals = next(iter(loader("test", COUNT)))[0][:COUNT]

rows = {"input": originals}
rows |= {name: reconstruct(model, originals, device) for name, model in models.items()}

figure = plot_image_rows(rows, title=f"reconstructions · {VARIANT} test")
figure.savefig(FIGURES / "vae_reconstructions.png", bbox_inches="tight")

## Latent traversals

The picture behind the locality number: hold one image's latent fixed and sweep a single
dimension. Every dimension gets a row, not just the first of each block -- on the
supervised model we know which rows *should* move the digit, but on the unsupervised one
the effect can sit anywhere, and only looking at all 16 shows where it actually went.

Row labels carry each dimension's block, which for the unsupervised model is nominal: the
same ranges the probes used, so the two figures line up row for row. The free dimensions
(10-15 here) are unclaimed in both, and are where anything the labels do not name should
end up.

The sweep runs over the same range for every row -- the 99th percentile of that model's
own encoded latents -- so a row that barely changes really is a dimension the decoder
ignores, not one that was swept too gently.

In [ ]:
STEPS = 7
source = originals[0]


def block_of(dim: int, blocks, names) -> str:
    """Which factor claims this dimension, or `free` if none does."""
    for block, factor in zip(blocks, names):
        if block.start <= dim < block.stop:
            return factor
    return "free"


for name, model in models.items():
    blocks, names = blocks_for(model, FALLBACK_BLOCKS, FACTOR_NAMES)
    span = float(np.percentile(np.abs(latents[name]), 99))
    values = np.linspace(-span, span, STEPS).round(2)

    dims = list(range(model.get_latent_dim().numel()))
    swept = latent_traversal(model, source, dims, values.tolist(), device)
    rows = {
        f"dim {dim} · {block_of(dim, blocks, names)}": images
        for dim, images in zip(dims, swept.values())
    }

    figure = plot_image_rows(
        rows,
        title=f"{name} — every latent dimension (sweep ±{span:.2f})",
        col_labels=values.tolist(),
    )
    figure.savefig(FIGURES / f"traversal_{name.replace(' ', '_')}.png", bbox_inches="tight")

## Reading the result

- **locality up, leakage down** — the block owns the factor. This is the precondition for
  marginalising a factor out by marginalising its dimensions.
- **locality up, leakage still ~1.0** — the heads made the factor readable in the block but
  did not remove it from the rest. Sufficient, not exclusive; a gradient-reversal head on
  the free dimensions is the next lever.
- **cost table roughly flat** — supervision was close to free. A large ELBO gap means gamma
  is too high and the heads are fighting the reconstruction.